# Stage 1 - Dataset processing (Kaggle)

Validate -> preprocess (loudness-normalise + resample + chunk) -> caption -> train/val split.

**Before running:**
1. Enable an accelerator is *not* required for Stage 1 (CPU only).
2. Add your raw audio as a **Kaggle Dataset** (mounts read-only under `/kaggle/input/<name>`).
3. Add this repository (upload it as a Dataset, or `git clone` in the first cell) and set `REPO_DIR`.

In [ ]:
import sys, os
from pathlib import Path

# Point this at the repo root (uploaded dataset or a git clone).
REPO_DIR = '/kaggle/working/metalcore'
if not Path(REPO_DIR).exists():
    # Fallback: clone your fork here.
    !git clone https://github.com/YOUR_USERNAME/metalcore.git {REPO_DIR}
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Repo:', REPO_DIR)

In [ ]:
!pip install -q -r requirements-dataset.txt

In [ ]:
# Set your input (raw audio) and output (processed dataset) paths.
INPUT_DIR = '/kaggle/input/YOUR_RAW_AUDIO_DATASET'   # <-- change me
OUTPUT_DIR = '/kaggle/working/dataset'
assert Path(INPUT_DIR).exists(), f'Input not found: {INPUT_DIR}'
print('Input :', INPUT_DIR)
print('Output:', OUTPUT_DIR)

In [ ]:
!python -m dataset_tools.cli all \
    --config configs/dataset.yaml \
    --input {INPUT_DIR} \
    --output {OUTPUT_DIR}

In [ ]:
# Inspect the result.
import json
meta = Path(OUTPUT_DIR) / 'metadata.jsonl'
lines = meta.read_text().splitlines()
print(f'{len(lines)} chunk(s). First record:')
print(json.dumps(json.loads(lines[0]), indent=2))
n_train = len((Path(OUTPUT_DIR) / 'train.jsonl').read_text().splitlines())
val_file = Path(OUTPUT_DIR) / 'val.jsonl'
n_val = len(val_file.read_text().splitlines()) if val_file.exists() else 0
print(f'train={n_train}  val={n_val}')

**Next:** package `OUTPUT_DIR` (or keep it in `/kaggle/working`) and use it in `02_music_lora_kaggle.ipynb`.
For persistence across sessions, save the processed dataset as a new Kaggle Dataset.